# Calibration ablation — **extend to CelebA** (append, don't overwrite Waterbirds)

Runs the identical calibration-policy ablation on the CelebA cells (resnet50_erm/celeba,
clip_vitb32/celeba), reusing the cached CelebA head posteriors (no retraining). calibration ∈
{marginal_split, mondrian, shift_robust} × method × score (APS/RAPS/THR) × ρ sweep × ≥3 seeds × ≥10
splits. **AFR is excluded on CelebA by the §2 worst-group-accuracy gate (worst-group collapse) — kept
excluded.**

`extend_ablation_to` MERGES into the existing `calibration_ablation.csv`: Waterbirds rows are kept,
CelebA rows appended/replaced; the MD + two figures per CelebA cell are regenerated from the full set.
Requires the Waterbirds ablation CSV to be present (Drive-persisted at `results/study/`). **STOP** after.

## 0. Parameters — **EDIT THESE**

In [ ]:
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/octadion/vgscp.git"
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
SEEDS         = 3
N_SPLITS      = 10
CELEBA_RESNET_MAX_TRAIN = 30000
CELEBA_SOURCE = "kaggle"   # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE  = ""
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + repo + CelebA data + Drive-backed results/study

In [ ]:
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from study_robust_train.colab_data import prepare_celeba
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
assert CELEBA_OK, "CelebA not available — set CELEBA_SOURCE/kaggle.json or CELEBA_DRIVE (this notebook is CelebA-only)."
os.environ["CELEBA_ROOT"] = CELEBA_ROOT; print("CelebA root:", CELEBA_ROOT)

# Drive-backed caches (feature cache hit from the grid run) + results/study (the persisted CSV)
for c in ("cache_clip", "cache_resnet", "study"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")
CSV = "results/study/calibration_ablation.csv"
print("existing ablation CSV present (Waterbirds to merge with):", os.path.exists(CSV))
if not os.path.exists(CSV):
    print("[note] No existing CSV -> this run will produce CelebA-only. Run the Waterbirds "
          "ablation (calibration_ablation.ipynb) first to MERGE both datasets.")

## 3. Build CelebA GridData (cache hit if the grid/recoverability already extracted features)

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_celeba():
    return {"dataset": {"root": os.environ["CELEBA_ROOT"], "n_classes": 2},
            "clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "max_train": CELEBA_RESNET_MAX_TRAIN, "cache_dir": "results/cache_resnet"}}

data, skipped = {}, []
for bb in ("resnet50_erm", "clip_vitb32"):
    try:
        t = time.time(); data[(bb, "celeba")] = build_griddata("celeba", bb, cfg_celeba(), seed=0)
        print(f"[built] {bb}/celeba ({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, "celeba")); print(f"[SKIP] {bb}/celeba: {e}")
print("CelebA cells:", list(data.keys()), "| skipped:", skipped)

## 4. Extend ablation to CelebA — append to CSV (Waterbirds kept), regenerate MD + figures → STOP

In [ ]:
from study_robust_train.calibration_ablation import extend_ablation_to

t = time.time()
out = extend_ablation_to(data, csv_path="results/study/calibration_ablation.csv",
                         md_path="CALIBRATION_ABLATION.md", figdir="results/study/figures",
                         seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)   # AFR excluded on CelebA by the §2 gate
print(f"[extend] merged in {(time.time()-t)/60:.1f} min; total cells now: {list(out['verdicts'].keys())}")
if out.get("new_excluded"):
    print("\nCelebA arms EXCLUDED (kept out — §2 worst-group acc gate; AFR expected):")
    for e in out["new_excluded"]:
        print(f"  {e['backbone']}/{e['dataset']} {e['method']} seed{e['seed']}: "
              f"worst-group acc {e['worst_group_acc']:.3f} < floor {e['floor']}")

for key in [k for k in out["verdicts"] if k[1] == "celeba"]:
    v = out["verdicts"][key]; c1, c2, c3 = v["C1"], v["C2"], v["C3"]
    print(f"\n{key}:")
    print(f"  C1 holds: {c1['C1_holds']}")
    for m, row in c1["methods"].items():
        print(f"    {m}: mondrian={row['mondrian']['mean']:.3f}  marginal_split={row['marginal_split']['mean']:.3f}"
              f"  (shortfall {row['split_shortfall']:+.3f})")
    print(f"  C2 efficiency tracks accuracy: {c2['efficiency_tracks_accuracy']} (corr {c2['acc_vs_setsize_corr']:.3f})")
    print(f"  C3 survives: {{m: {{sc: c3['methods'][m][sc]['survives'] for sc in c3['methods'][m]}} for m in c3['methods']}}")

## 5. Show updated CALIBRATION_ABLATION.md + CelebA figures

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("CALIBRATION_ABLATION.md", encoding="utf-8").read()))
import glob
for p in sorted(glob.glob("results/study/figures/calib_C1_*celeba.png")):
    display(Image(p))

## 6. STOP — CelebA ablation merged
`CALIBRATION_ABLATION.md` + `calibration_ablation.csv` now hold Waterbirds **and** CelebA (Waterbirds
untouched). Two C1 figures per CelebA cell written. Hand C1/C2/C3 (both datasets) to the researcher.
**STOP for human review** — do NOT run the full-GroupDRO fine-tune or any 3rd/4th dataset.